# TurtleBot3 Semantic Navigation — GPU host driver

Runs the whole stack (Gazebo Harmonic + SLAM + Nav2 + YOLOv8n + semantic memory +
LocateAnything grounding) on this machine's NVIDIA GPU and drives it from here.
There is no RViz and no Gazebo GUI: the container is headless, gz-sim renders its
camera through EGL straight on the GPU, and the map / camera / landmark views are
rendered inline below.

**Order matters.** Run the setup and build sections once, restart the kernel, then
work through Launch → Connect → Look → Command. The launch has a ~60 s staged
startup ladder (Gazebo → Nav2+SLAM → perception → coordinator) and the readiness
cell waits it out for you.

Helper code lives in [`tb3_nb.py`](tb3_nb.py) next to this notebook — read it when
you want to know what a cell is actually doing.

## 1. Environment check

Confirms the container can see the GPU and that torch is a CUDA build. If
`torch.cuda` is unavailable here, everything below still runs — just slowly, on
CPU — which is a confusing failure to debug later, so check it now.

In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath("tb3_nb.py")))

import tb3_nb
from tb3_nb import Stack, Ros, show_camera, show_map, watch_navigation, grounding_health

env = tb3_nb.check_env()

## 2. One-time build

Fetches the YOLOv8n weights (git-ignored, ~6 MB) and builds the colcon workspace.
Takes a few minutes the first time and is incremental afterwards.

**Restart the kernel after this cell finishes** so the freshly built `install/`
is on the Python path — otherwise importing the workspace's message types in
later cells picks up a stale tree.

Skip this section entirely if `check_env` above already reported
`workspace built ✓`.

In [ ]:
tb3_nb.fetch_weights()
tb3_nb.build_workspace()

## 3. Launch the stack

`Stack` runs `ros2 launch` detached in its own process group, logging to
`log/notebook/stack.log`. Detached because the kernel outlives any single cell,
and to a file because a cell that stops draining a pipe would eventually block
every node in the stack on its next log write.

Worlds (`world=`):

| alias | size | contents |
| --- | --- | --- |
| `warehouse_models_person` | 6×6 m | **default** — five `person` figures, for `go to person N` |
| `warehouse_models` | 4×6 m | one `table` (alias `bench`) + one `person` |
| `warehouse_aws` | 8×6 m | AWS warehouse props, chairs, orange sofa — built for grounding queries |

In [ ]:
stack = Stack(world="warehouse_models_person")
stack.start()

Watch the startup for a bit. Gazebo loading meshes and Nav2's lifecycle managers
walking every node through configure/activate are what you should see here.

In [ ]:
stack.follow(seconds=30)

## 4. Connect and wait for readiness

`Ros` is one rclpy node spinning on a background thread, caching the newest
message per topic. Cells then read state synchronously instead of each spinning
its own node.

`wait_until_ready()` blocks until the stack is genuinely usable — SLAM publishing
a map, camera frames flowing, detector alive, coordinator announcing a state —
rather than merely until the launch process exists.

In [ ]:
ros = Ros()
ros.wait_until_ready(timeout=180)

## 5. What the robot sees

The annotated detector frame — YOLOv8n boxes drawn over the camera image. This is
the `~/debug_image` topic RViz used to display.

`detector_node` only publishes it while something is subscribed, so the very
first call may come back empty; re-run the cell.

In [ ]:
show_camera(ros, debug=True)
ros.detections()

## 6. The SLAM map and semantic landmarks

The inline replacement for the RViz view: occupancy grid, persistent semantic
landmarks (red, labelled with their memory ids), and the robot's pose from TF
(blue). Re-run it as exploration proceeds to watch the map fill in.

Landmark ids like `person_3` are **memory slots assigned in observation order**,
not identity recognition — the same physical figure can be a different index
across runs depending on the path the robot took.

In [ ]:
show_map(ros)
ros.landmarks()

Let exploration run for a while and look again — frontier exploration drives the
robot around on its own until you send a command.

In [ ]:
import time
time.sleep(60)
show_map(ros)

## 7. Send a semantic command

`go to person` picks the **nearest** observed person; `go to person 3` picks the
specific memory slot. The parser is deterministic and rule-based — no LLM. The
coordinator pauses exploration, resolves the target, drives Nav2 to a standoff
pose, then resumes exploring.

Only targets already in semantic memory can be reached: if the robot has not seen
a person yet, the query fails. Check `ros.landmarks()` first.

In [ ]:
ros.send_command("go to person")

Follow the coordinator's state machine
(`EXPLORING → SEMANTIC_QUERYING → SEMANTIC_NAV → TARGET_REACHED / FAILED`).

In [ ]:
watch_navigation(ros, seconds=120)

In [ ]:
show_map(ros)
show_camera(ros)

## 8. Attribute queries (LocateAnything)

Commands carrying **descriptive attributes** are resolved by NVIDIA
LocateAnything-3B running in the `grounding` sidecar container, which shares this
container's network namespace (hence `127.0.0.1:8801`). The query node sends each
same-class candidate's stored best-view frame plus the phrase, and scores
candidates by IoU between the model's box and the candidate's own bbox.

Check which backend actually loaded first — the `mock` backend answers colour
words only, via an HSV heuristic, so a failure means something different in each
case. The real weights are ~7 GB and download on the sidecar's first start.

Use the `warehouse_aws` world for this: it is the one with an orange sofa and
coloured chairs to disambiguate between.

In [ ]:
grounding_health()

In [ ]:
ros.send_command("go to the sofa with warm color")
watch_navigation(ros, seconds=120)
print("query status:", ros.query_status())

## 9. Troubleshooting

The launch log is the first place to look. Useful filters:

- `grep="error"` — anything that failed outright
- `grep="detector_node"` — confirm it reports `device=cuda:0`; `device=cpu` means
  the container did not get the GPU
- `grep="lifecycle"` / `grep="Nav2"` — a node stuck `inactive` means a lifecycle
  transition response was dropped and the startup ladder needs more slack
- `grep="frontier"` — exploration stalls usually show up here as rejected goals

In [ ]:
stack.logs(60, grep="error")

In [ ]:
stack.logs(30, grep="detector_node")

In [ ]:
# Everything the stack is publishing right now.
ros.topics()

## 10. Shut down

`stop()` SIGINTs the whole process group. Always run it before relaunching —
leftover gz-sim, Nav2 and SLAM processes will fight the next launch over topic
names and the sim clock, which presents as a stack that comes up but never moves.

In [ ]:
stack.stop()
ros.shutdown()